<a href="https://colab.research.google.com/github/Zafeerxyz/Machine-Learning-algorithms/blob/main/deliveryDelay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd

df = pd.read_csv("delivery_delay_practice_dataset.csv")

print(df.shape)
print(df.head())
print(df.isnull().sum())
print(df["is_delayed"].value_counts())

(1200, 10)
   order_id  distance_km traffic_level weather  preparation_time vehicle_type  \
0  ORD00001         7.76           Low   Clear              17.9         Bike   
1  ORD00002        10.30           Low   Clear               6.9      Scooter   
2  ORD00003         6.88        Medium   Clear               9.9         Bike   
3  ORD00004         6.22          High   Clear              10.1         Bike   
4  ORD00005        11.12        Medium    Rain              23.3         Bike   

   hour_of_day  is_weekend  delivery_minutes  is_delayed  
0           16           0              52.2           1  
1           12           0              38.2           0  
2            9           1              35.7           0  
3           14           1              59.5           1  
4           18           1              81.0           1  
order_id             0
distance_km          0
traffic_level       18
weather             30
preparation_time    24
vehicle_type         0
hour_of_da

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

X = df.drop(columns = "is_delayed")
y = df["is_delayed"]

numerical_features = ["distance_km","preparation_time","hour_of_day"]

categorical_features = ["traffic_level","weather","vehicle_type"]

numerical_pipeline = Pipeline(steps = [("impute",SimpleImputer(strategy="median"))
                                        ])

categorical_pipeline = Pipeline(steps = [("imputer",SimpleImputer(strategy="most_frequent")),
                                         ("encoder",OneHotEncoder(handle_unknown= "ignore"))
                                         ])
transformer = ColumnTransformer(transformers=[("num",numerical_pipeline,numerical_features),
                                              ("cat",categorical_pipeline,categorical_features)])

full_pipeline = Pipeline(steps = [("transformer",transformer),
                                  ("model",RandomForestClassifier(random_state=42,
                                                                  n_estimators=200,
                                                                  class_weight="balanced"))])

X_train , X_test, y_train, y_test = train_test_split(X, y , random_state = 42 ,stratify=y)

full_pipeline.fit(X_train,y_train)

from sklearn.metrics import accuracy_score,f1_score,classification_report

pred = full_pipeline.predict(X_test)

accuracy = accuracy_score(y_test,pred)
print("Accuracy:",accuracy)
print(classification_report(y_test, pred))

Accuracy: 0.8933333333333333
              precision    recall  f1-score   support

           0       0.86      0.76      0.80        86
           1       0.91      0.95      0.93       214

    accuracy                           0.89       300
   macro avg       0.88      0.85      0.86       300
weighted avg       0.89      0.89      0.89       300



In [18]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 5]
}



grid_search = GridSearchCV(
    estimator = full_pipeline,
    param_grid= param_grid,
    verbose=1,
    scoring= "f1",
    cv =5,
    n_jobs =-1
)


grid_search.fit(X_train,y_train)

print("best parameter:",grid_search.best_params_)
print("best cv:",grid_search.best_score_)

best_model = grid_search.best_estimator_

test_prediction = best_model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_prediction))
print("Test F1:", f1_score(y_test, test_prediction))
print(classification_report(y_test, test_prediction))


Fitting 5 folds for each of 18 candidates, totalling 90 fits
best parameter: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
best cv: 0.9029235565130573
Test accuracy: 0.9033333333333333
Test F1: 0.9327146171693735
              precision    recall  f1-score   support

           0       0.84      0.81      0.83        86
           1       0.93      0.94      0.93       214

    accuracy                           0.90       300
   macro avg       0.88      0.88      0.88       300
weighted avg       0.90      0.90      0.90       300

